In [7]:
import pyodbc

# اتصال به SQL Server
conn = pyodbc.connect(
    'DRIVER={SQL Server};'
    'SERVER=MKZ-DSAS\\DSAS;'
    'DATABASE=DSAS;'
    'UID=datadriven;'
    'PWD=5Rdx@4Rfv1355'
)

cursor = conn.cursor()

# لیست AssetIDهایی که می‌خوای بررسی بشن
asset_ids = [8341, 8342, 8343, 8344, 8346, 9286, 9287]

# لیست برای ذخیره مقادیر Value
values = []

# اجرای کوئری برای هر AssetID و دریافت فقط Value
for asset_id in asset_ids:
    query = f"""
        SELECT TOP 1 [Value]
        FROM [PDA].[Periodic_Values]
        WHERE UnitID = 11 AND AssetID = {asset_id}
        ORDER BY DateTime DESC
    """
    cursor.execute(query)
    row = cursor.fetchone()
    if row:
        values.append(row.Value)
    else:
        values.append(None)  


value_8341, value_8342, value_8343, value_8344, value_8346, value_9286, value_9287 = values

# نمایش مقادیر
print("✅ مقادیر آخرین رکوردها برای UnitID=11:")
print(f"AssetID 8341 → Value: {value_8341}")
print(f"AssetID 8342 → Value: {value_8342}")
print(f"AssetID 8343 → Value: {value_8343}")
print(f"AssetID 8344 → Value: {value_8344}")
print(f"AssetID 8346 → Value: {value_8346}")
print(f"AssetID 9286 → Value: {value_9286}")
print(f"AssetID 9287 → Value: {value_9287}")

# بستن اتصال
cursor.close()
conn.close()




import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances
import joblib

# بارگذاری اجزای مدل
scaler = joblib.load('scaler.pkl')
dbscan = joblib.load('dbscan_model.pkl')
cluster_points = np.load('cluster_points.npy')

# تابع تشخیص ناهنجاری با آستانه وزن
def is_anomalous(input_dict, threshold=10.0):
    input_df = pd.DataFrame([input_dict])
    scaled_input = scaler.transform(input_df)
    label = dbscan.fit_predict(scaled_input)[0]

    if label != -1:
        return {'is_anomaly': False, 'anomaly_weight': 0.0}

    # محاسبه فاصله از نزدیک‌ترین خوشه
    distance = pairwise_distances(scaled_input, cluster_points).min()
    anomaly_weight = distance

    # بررسی آستانه
    is_anomaly = anomaly_weight > threshold
    return {'is_anomaly': is_anomaly, 'anomaly_weight': anomaly_weight}

# مثال استفاده
sample_input = {
    'AssetID_8341': value_8341,
    'AssetID_8342': value_8342,
    'AssetID_8343': value_8343,
    'AssetID_8344': value_8344,
    'AssetID_8346': value_8346,
    'AssetID_9286': value_9286,
    'AssetID_9287': value_9287
}

result = is_anomalous(sample_input)
print(result)



import mysql.connector
from datetime import datetime

# اتصال به دیتابیس MySQL
conn = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password='',  
    database='dsas'
)

cursor = conn.cursor()

# مقادیر ورودی
inputs = [value_8341,value_8342,value_8343,value_8344,value_8346,value_9286,value_9287]
anomaly_weight = result['anomaly_weight']  # مقدار score
results = "Normal" if anomaly_weight < 5 else "Abnormal"
model_name = "Anomaly detection for lube oil system"
unitID = 11
system = "dbscan clustering weighted by computing distance from clusters "
score = anomaly_weight
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
updated_at = created_at


query = """
    INSERT INTO results_dsas_mhi_lube_oil_11 
    (inputs, results, model_name, unitID, system, score, created_at, updated_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""


cursor.execute(query, (
    str(inputs),  # تبدیل لیست به رشته برای ذخیره در فیلد text یا varchar
    results,
    model_name,
    unitID,
    system,
    score,
    created_at,
    updated_at
))

# ذخیره تغییرات
conn.commit()

print("✅ داده با موفقیت ثبت شد.")

# بستن اتصال
cursor.close()
conn.close()


✅ مقادیر آخرین رکوردها برای UnitID=11:
AssetID 8341 → Value: 0.2
AssetID 8342 → Value: 10.9
AssetID 8343 → Value: 68.0
AssetID 8344 → Value: -230.0
AssetID 8346 → Value: 6.0
AssetID 9286 → Value: 7.5
AssetID 9287 → Value: 1.21


c:\Users\pishva_r\Anaconda3\envs\test2\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\pishva_r\Anaconda3\envs\test2\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DBSCAN from version 1.5.1 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


{'is_anomaly': False, 'anomaly_weight': 1.389397787167795}


DatabaseError: 2003 (HY000): Can't connect to MySQL server on '127.0.0.1:3306' (10061)

In [1]:
import pandas as pd

# مسیر فایل اکسل
file_path = 'test.xlsx'  # ← اینجا نام فایل خودت رو بذار

# خواندن فایل اکسل
df = pd.read_excel(file_path)

# فرض بر اینه که ستون A بدون نام خاصی هست، پس از طریق اندیس صفر بهش دسترسی داریم
column_values = df.iloc[:, 0]  # ستون اول

# ساخت رشته نهایی با شماره ردیف و علامت "-"
combined_text = ''
for i, value in enumerate(column_values, start=1):
    combined_text += f'{i}-{value} '

# حذف فاصله‌ی اضافی آخر
combined_text = combined_text.strip()

# نمایش خروجی
print(combined_text)


1-درصد تبدیل نقاط قابل بهبود شناسایی شده به
 پروژه‌های بهبود سازمان (درصد) 2-درصد پیشرفت پروژه‌های بهبود سازمانی تعریف شده (درصد) 3-درصد اقدامات اصلاحی
 اجرا شده در موعد مقرر (درصد) 4-درصد اقدامات اصلاحی
 فاقد اثربخشی
 (درصد) 5-درصد مستندات 
بازنگری شده (درصد) 6-مدت‌زمان بازنگری مستندات (روز) 7-مدت‌زمان تهیه مستندات (روز) 8-درصد تعیین
 تکلیف سوابق 9-درصد اثربخشی خدمات نامنطبق رفع شده  10-متوسط زمان رفع ریشه ای خدمات نامنطبق  11-درصد اجرای مصوبات بازنگری مدیریت (درصد) 12-درصد تغییرات
 اجرا شده  13-درصد تغییرات
 اثربخش  14-درصد کاهش ریسک فرایندی (درصد) 15-درصد ریسک‌های فرایندی به‌روز شده (درصد) 16-درصد دانش‌های 
ارزیابی شده (درصد) 17-درصد دانش‌های 
کاربردی شده (درصد) 18-مدت‌زمان ارزیابی درخواست‌های دانش ثبت شده (درصد) 19-درصد تحقق درآمد (درصد) 20-درصد تحقق فروش (درصد) 21-ضریب خروج برنامه‌ریزی نشده ناشی (در تعهد بهره‌بردار) 22-نرخ خروج داخلی 23- نرخ خروج اضطراری 24-نرخ خروج با هماهنگی 25-نرخ خروج افزایش اجباری 
زمان تعمیرات 26-آمادگی محقق شده به آمادگی قراردادی (درصد) 27-آمادگی محقق شده ب

In [2]:
# pip install pandas openpyxl

In [8]:
import pandas as pd

# بارگذاری داده‌ها از شیت 1، شروع از ردیف 4
file_path = "kpis.xlsx"
df = pd.read_excel(file_path, sheet_name="1", header=None, skiprows=3, usecols="A:B")

# حذف ردیف‌هایی که شماره یا عنوان ندارن
df = df.dropna(subset=[0, 1])

# ترکیب شماره و عنوان شاخص به صورت سطری
for index, row in df.iterrows():
    print(f"{int(row[0])} - {row[1]}")


1 - اندازه‌گیری بموقع شاخص‌های فرایندی (روز)
2 - درصد تبدیل نقاط قابل بهبود شناسایی شده به
 پروژه‌های بهبود سازمان (درصد)
3 - درصد پیشرفت پروژه‌های بهبود سازمانی تعریف شده (درصد)
4 - درصد اقدامات اصلاحی
 اجرا شده در موعد مقرر (درصد)
5 - درصد اقدامات اصلاحی
 فاقد اثربخشی
 (درصد)
6 - درصد مستندات 
بازنگری شده (درصد)
7 - مدت‌زمان بازنگری مستندات (روز)
8 - مدت‌زمان تهیه مستندات (روز)
9 - درصد تعیین
 تکلیف سوابق
10 - درصد اثربخشی خدمات نامنطبق رفع شده 
11 - متوسط زمان رفع ریشه ای خدمات نامنطبق 
12 - درصد اجرای مصوبات بازنگری مدیریت (درصد)
13 - درصد تغییرات
 اجرا شده 
14 - درصد تغییرات
 اثربخش 
15 - درصد کاهش ریسک فرایندی (درصد)
16 - درصد ریسک‌های فرایندی به‌روز شده (درصد)
17 - درصد دانش‌های 
ارزیابی شده (درصد)
18 - درصد دانش‌های 
کاربردی شده (درصد)
19 - مدت‌زمان ارزیابی درخواست‌های دانش ثبت شده (درصد)
20 - درصد تحقق درآمد (درصد)
21 - درصد تحقق فروش (درصد)
22 - ضریب خروج برنامه‌ریزی نشده ناشی (در تعهد بهره‌بردار)
23 - نرخ خروج داخلی
24 -  نرخ خروج اضطراری
25 - نرخ خروج با هماهنگی
26 - نرخ خر

In [9]:
import openpyxl

# بارگذاری فایل اکسل
wb = openpyxl.load_workbook("kpis.xlsx")

# استخراج داده‌های شیت 1: شماره شاخص در ستون A، فرمول در ستون C
sheet1 = wb["1"]
formula_map = {}
for row in sheet1.iter_rows(min_row=2, values_only=True):
    index = row[0]  # ستون A
    formula = row[2]  # ستون C
    if index is not None:
        formula_map[str(index).strip()] = formula

# پردازش شیت‌های 2 تا 8
for i in range(2, 9):
    sheet = wb[str(i)]
    
    # اضافه کردن عنوان ستون جدید در سلول D1
    sheet["D1"] = "فرمول شاخص"
    
    # پیمایش ردیف‌ها از B2 به بعد
    for row in range(2, sheet.max_row + 1):
        index_cell = sheet[f"B{row}"]
        formula_cell = sheet[f"D{row}"]
        
        index_value = str(index_cell.value).strip() if index_cell.value is not None else ""
        formula_value = formula_map.get(index_value, "")
        
        formula_cell.value = formula_value

# ذخیره فایل
wb.save("kpis_updated.xlsx")


In [ ]:
import pyodbc

# مشخصات اتصال
server = 'MKZ-DSAS\\DSAS'  # نام سرور و
database = 'DSAS'          # نام دیتابیس
username = 'datadriven'            # یوزر فرضی (جایگزین کن)
password = '5Rdx@4Rfv1355'  # پسورد فرضی (جایگزین کن)

# رشته اتصال
conn_str = (
    'DRIVER={SQL Server};'
    'SERVER=MKZ-DSAS\\DSAS;'
    'DATABASE=DSAS;'
    'UID=datadriven;'
    'PWD=5Rdx@4Rfv1355'
)


try:
    # تلاش برای اتصال
    conn = pyodbc.connect(conn_str)
    print("اتصال موفق بود!")
    conn.close()  # بستن اتصال بعد از تست
except pyodbc.Error as e:
    print(f"خطا در اتصال: {e}")

اتصال موفق بود!


In [4]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'PostgreSQL ANSI(x64)', 'PostgreSQL Unicode(x64)', 'Amazon Redshift (x64)', 'SQL Server Native Client 10.0', 'MySQL ODBC 5.3 ANSI Driver', 'MySQL ODBC 5.3 Unicode Driver', 'ODBC Driver 18 for SQL Server']
